In [ ]:
# Standard library imports
import os
import sys
import string
import importlib
from collections import Counter

# Third-party imports
import cv2
import numpy as np
import matplotlib.pyplot as plt
import fitz
import spacy
import openpyxl
from PIL import Image, ImageDraw, ImageFont

# Local imports
sys.path.insert(0, os.getcwd())
import overlay_function
importlib.reload(overlay_function)
from overlay_function import overlay_names_on_redaction_boxes

In [ ]:
# Configuration
PDF_FOLDER = os.path.join(os.getcwd(), "PDFS")
PDF_NAME = "vol00008-official-doj-latest-efta00037366.pdf"
DPI = 300

DOC = fitz.open(os.path.join(PDF_FOLDER, PDF_NAME))
pdf_path = os.path.join(PDF_FOLDER, PDF_NAME)
font_path = os.path.join(os.getcwd(), "fonts", "timesnewromanpsmt.ttf")

In [9]:
def extract_redaction_boxes(pdf, black_threshold=100, min_width=50, min_height=30, min_aspect_ratio=1.5, visualize=False, output_dir=None):
    """Extract all types of redaction boxes from PDF."""
    boxes = []
    scale = DPI / 96.0

    doc_number = None
    first_page = pdf[0]
    page_text = first_page.get_text()
    for line in page_text.split('\n'):
        line = line.strip()
        if 'EFTA' in line:
            import re
            match = re.search(r'EFTA\d+', line)
            if match:
                doc_number = match.group(0)
                break

    if doc_number is None:
        doc_number = "UNKNOWN"

    boxes_per_page = {}

    for page_index in range(len(pdf)):
        page = pdf[page_index]
        boxes_per_page[page_index] = 0

        mat = fitz.Matrix(scale, scale)
        pix = page.get_pixmap(matrix=mat)
        img_array = np.frombuffer(pix.samples, dtype=np.uint8)

        if pix.n >= 3:
            img_array = img_array.reshape(pix.h, pix.w, pix.n)
            img_array = img_array[:, :, :3]
            img = cv2.cvtColor(img_array, cv2.COLOR_RGB2BGR)
        elif pix.n == 1:
            img_array = img_array.reshape(pix.h, pix.w)
            img = cv2.cvtColor(img_array, cv2.COLOR_GRAY2BGR)
        else:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # METHOD 1: Solid black boxes
        _, raw_thresh = cv2.threshold(gray, black_threshold, 255, cv2.THRESH_BINARY_INV)
        kernel = np.ones((1, 9), np.uint8)
        thresh = cv2.dilate(raw_thresh, kernel, iterations=1)

        def refine_bbox_to_raw(x, y, w, h):
            pad = 2
            rx1 = max(0, x - pad)
            ry1 = max(0, y - pad)
            rx2 = min(raw_thresh.shape[1], x + w + pad)
            ry2 = min(raw_thresh.shape[0], y + h + pad)

            if rx2 <= rx1 or ry2 <= ry1:
                return x, y, w, h

            roi = raw_thresh[ry1:ry2, rx1:rx2]
            roi_contours, _ = cv2.findContours(roi, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if not roi_contours:
                return x, y, w, h

            cx_local = (x + w / 2.0) - rx1
            cy_local = (y + h / 2.0) - ry1

            best_contour = None
            best_score = float('-inf')
            for c in roi_contours:
                area = cv2.contourArea(c)
                if area < 8:
                    continue

                contains_center = cv2.pointPolygonTest(c, (cx_local, cy_local), False) >= 0
                bx, by, bw, bh = cv2.boundingRect(c)
                ccx = bx + bw / 2.0
                ccy = by + bh / 2.0
                dist2 = (ccx - cx_local) ** 2 + (ccy - cy_local) ** 2

                score = area * (3.0 if contains_center else 1.0) - 0.15 * dist2
                if score > best_score:
                    best_score = score
                    best_contour = c

            if best_contour is None:
                return x, y, w, h

            bx, by, bw, bh = cv2.boundingRect(best_contour)
            return rx1 + bx, ry1 + by, bw, bh

        def looks_like_punctuation_candidate(x, y, w, h, binary_mask):
            """Reject punctuation artifacts (e.g., quotes/colons) that overlap real boxes."""
            if w <= 0 or h <= 0:
                return True

            # Strong prior for punctuation-sized marks
            if w < 14 or h < 10:
                return True

            aspect_ratio = w / float(max(1, h))
            if aspect_ratio < 0.42:
                return True

            roi = binary_mask[y:y + h, x:x + w]
            if roi.size == 0:
                return True

            ink_ratio = float(np.count_nonzero(roi)) / float(max(1, w * h))

            # Reject small isolated text characters (letters have hollow centers or low ink density)
            # Text characters typically are 8-20px wide and 10-28px tall with <0.50 ink
            if w <= 20 and h <= 30 and ink_ratio < 0.50:
                return True
            
            # Additional check: very small shapes with moderate ink are probably punctuation
            if w <= 16 and h <= 20 and ink_ratio < 0.55:
                return True
            
            # Question marks are sparse overall (dot + curved line) - reject if sparse small shape
            if w <= 22 and h <= 35 and ink_ratio < 0.38:  # Very sparse - likely punctuation
                return True
            if w <= 34 and h <= 42:
                col_fill = np.sum(roi > 0, axis=0) / float(max(1, h))
                row_fill = np.sum(roi > 0, axis=1) / float(max(1, w))
                dense_col_ratio = float(np.mean(col_fill >= 0.45)) if col_fill.size else 0.0
                dense_row_ratio = float(np.mean(row_fill >= 0.45)) if row_fill.size else 0.0

                # Reject as punctuation only if sparse AND not a solid block. Solid small boxes are redactions.
                if ((dense_col_ratio < 0.28 and dense_row_ratio < 0.48) or (ink_ratio < 0.32 and w < 28)) and ink_ratio < 0.65:
                    return True
            # Check for a solid dense row (main redaction bar). If it exists, it's a redaction not punctuation.
            row_blackness = np.sum(roi < 200, axis=1)  # Count dark pixels per row
            max_row_blackness = float(np.max(row_blackness)) if row_blackness.size > 0 else 0.0
            max_possible_row_blackness = float(w)

            bbox_area = float(max(1, w * h))
            
            # Use connected components to analyze blob structure
            roi_uint8 = roi.astype(np.uint8)
            num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(roi_uint8, connectivity=8)
            
            small_blob_centers = []
            for idx in range(1, num_labels):
                c_area = stats[idx, cv2.CC_STAT_AREA]
                c_w = stats[idx, cv2.CC_STAT_WIDTH]
                c_h = stats[idx, cv2.CC_STAT_HEIGHT]

                if (c_area <= bbox_area * 0.22 and
                    c_w <= max(9, int(w * 0.72)) and
                    c_h <= max(12, int(h * 0.58))):
                    small_blob_centers.append(float(centroids[idx][1]))

                # Dot/tail component near bottom suggests comma, semicolon, or question mark artifact
                if w <= 35 and h <= 48 and c_area <= bbox_area * 0.12:  # Increased dot area threshold and size range
                    cy = float(centroids[idx][1])
                    if cy >= h * 0.60:  # Lowered threshold - dot in bottom 40%
                        # For question marks: if overall shape is very sparse, reject it
                        if h >= 18 and ink_ratio < 0.40:  # Question marks are taller but sparse
                            return True
                        # For comma/semicolon: if it's small, reject it
                        if w <= 30 and h <= 44 and ink_ratio < 0.55:
                            return True

            if len(small_blob_centers) >= 2 and w <= 26:
                vertical_spread = max(small_blob_centers) - min(small_blob_centers)
                if vertical_spread >= max(5.0, h * 0.28):
                    return True

            return False

        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contours = sorted(contours, key=lambda c: (cv2.boundingRect(c)[1], cv2.boundingRect(c)[0]))

        method1_boxes = []
        for contour in contours:
            x, y, w, h = cv2.boundingRect(contour)

            x, y, w, h = refine_bbox_to_raw(x, y, w, h)

            if w <= 0 or h <= 0:
                continue

            bbox_area = w * h
            if bbox_area <= 0:
                continue

            aspect_ratio = w / float(h)
            fill_ratio = np.count_nonzero(raw_thresh[y:y + h, x:x + w]) / float(bbox_area)

            # Relaxed but controlled constraints: catch small boxes without swallowing nearby words
            if not (bbox_area > 20 and fill_ratio > 0.32 and w >= 8 and h >= 10 and aspect_ratio >= 0.50):
                continue

            # Check interior for white pixels (background/text)
            margin = 1
            interior_y1 = max(0, y + margin)
            interior_y2 = min(gray.shape[0], y + h - margin)
            interior_x1 = max(0, x + margin)
            interior_x2 = min(gray.shape[1], x + w - margin)

            if interior_y2 > interior_y1 and interior_x2 > interior_x1:
                interior = gray[interior_y1:interior_y2, interior_x1:interior_x2]
                white_pixels = np.sum(interior > 220)
                white_ratio = white_pixels / float(interior.size)
                
                # Check lower portion of box (should be solid black redaction, not text)
                lower_half_y = int(interior.shape[0] * 0.5)
                lower_half = interior[lower_half_y:, :]
                if lower_half.size > 0:
                    lower_white_ratio = np.sum(lower_half > 220) / float(lower_half.size)
                else:
                    lower_white_ratio = 1.0
            else:
                white_ratio = 0
                lower_white_ratio = 0

            # Accept based on size-adaptive thresholds (larger boxes can have more grey border)
            bbox_size = max(w, h)
            if bbox_size > 200:
                # Large boxes: allow up to 50% lighter pixels (grey borders OK)
                accept_box = white_ratio < 0.50
            elif bbox_size > 80:
                # Medium boxes: allow up to 45% white (relaxed for grey borders)
                accept_box = white_ratio < 0.45 or (white_ratio < 0.50 and lower_white_ratio < 0.15)
            else:
                # Small boxes: stricter threshold
                accept_box = white_ratio < 0.30 or (white_ratio < 0.45 and lower_white_ratio < 0.12)
            
            if accept_box:
                if looks_like_punctuation_candidate(x, y, w, h, raw_thresh):
                    continue
                method1_boxes.append({
                    'x': x, 'y': y, 'w': w, 'h': h, 'method': 1
                })

        # METHOD 2: Hough line detection
        edges = cv2.Canny(gray, 100, 100)
        lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=50, minLineLength=40, maxLineGap=15)

        method2_boxes = []
        if lines is not None:
            h_lines = []
            v_lines = []
            for line in lines:
                x1, y1, x2, y2 = line[0]
                angle = abs(np.arctan2(y2 - y1, x2 - x1))
                if angle < 0.3 or angle > np.pi - 0.3:
                    h_lines.append(((min(x1, x2), max(x1, x2), (y1 + y2) // 2), line[0]))
                elif 1.24 < angle < 1.9:
                    v_lines.append(((min(y1, y2), max(y1, y2), (x1 + x2) // 2), line[0]))

            for h_idx, (h_info, h_line) in enumerate(h_lines[:]):
                h_left, h_right, h_y = h_info
                h_len = h_right - h_left
                if h_len < min_width:
                    continue

                for v_idx, (v_info, v_line) in enumerate(v_lines[:]):
                    v_top, v_bottom, v_x = v_info
                    v_len = v_bottom - v_top
                    if v_len < min_height:
                        continue

                    if (abs(v_x - h_left) < 5 or abs(v_x - h_right) < 5) and abs(h_y - v_top) < 5:
                        box_x = min(h_left, v_x)
                        box_y = v_top
                        box_w = h_len
                        box_h = v_len

                        # Check interior for white pixels
                        margin = 2
                        interior_y1 = max(0, box_y + margin)
                        interior_y2 = min(gray.shape[0], box_y + box_h - margin)
                        interior_x1 = max(0, box_x + margin)
                        interior_x2 = min(gray.shape[1], box_x + box_w - margin)

                        if interior_y2 > interior_y1 and interior_x2 > interior_x1:
                            interior = gray[interior_y1:interior_y2, interior_x1:interior_x2]
                            white_pixels = np.sum(interior > 220)
                            white_ratio = white_pixels / float(interior.size)
                        else:
                            white_ratio = 0

                        # Only add if low white pixels
                        if white_ratio < 0.10:
                            if looks_like_punctuation_candidate(box_x, box_y, box_w, box_h, raw_thresh):
                                continue
                            method2_boxes.append({
                                'x': box_x, 'y': box_y, 'w': box_w, 'h': box_h, 'method': 2
                            })

        # METHOD 3: Grey-edged boxes (must have visible grey shading, not pure black)
        _, thresh_grey = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY_INV)
        contours_grey, _ = cv2.findContours(thresh_grey, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        method3_boxes = []
        for contour in contours_grey:
            x, y, w, h = cv2.boundingRect(contour)

            # Relaxed size constraints to catch smaller grey-bordered boxes
            if w < 20 or h < 20 or h > 55:
                continue

            aspect_ratio = w / float(h) if h > 0 else 0
            if aspect_ratio < 0.9:
                continue

            # Check interior: must have grey shading (100-200 range), not pure black
            margin = 2
            interior_y1 = max(0, y + margin)
            interior_y2 = min(gray.shape[0], y + h - margin)
            interior_x1 = max(0, x + margin)
            interior_x2 = min(gray.shape[1], x + w - margin)

            if interior_y2 <= interior_y1 or interior_x2 <= interior_x1:
                continue

            interior = gray[interior_y1:interior_y2, interior_x1:interior_x2]

            white_pixels = np.sum(interior > 220)
            dark_pixels = np.sum(interior < 100)
            grey_pixels = np.sum((interior >= 100) & (interior <= 220))
            total_pixels = interior.size

            white_ratio = white_pixels / float(total_pixels) if total_pixels > 0 else 1.0
            grey_ratio = grey_pixels / float(total_pixels) if total_pixels > 0 else 0.0
            dark_ratio = dark_pixels / float(total_pixels) if total_pixels > 0 else 0.0

            # Accept only if has visible grey content, low white, AND not mostly solid black
            if grey_ratio > 0.20 and white_ratio < 0.15 and dark_ratio < 0.55:
                is_duplicate = False
                for m1_box in method1_boxes:
                    if abs(m1_box['y'] - y) < 5 and abs(m1_box['x'] - x) < 10:
                        is_duplicate = True
                        break

                if not is_duplicate:
                    if looks_like_punctuation_candidate(x, y, w, h, raw_thresh):
                        continue
                    method3_boxes.append({
                        'x': x, 'y': y, 'w': w, 'h': h, 'method': 3
                    })

        # Deduplicate M1/M2, add all M3
        m1_m2_combined = []
        used = set()
        all_m1_m2 = method1_boxes + method2_boxes

        for i, box in enumerate(all_m1_m2):
            if i in used:
                continue
            m1_m2_combined.append(box)
            for j in range(i + 1, len(all_m1_m2)):
                if j not in used:
                    other = all_m1_m2[j]
                    if (abs(box['x'] - other['x']) < 20 and
                        abs(box['y'] - other['y']) < 8 and
                        abs(box['w'] - other['w']) < 50):
                        used.add(j)

        final_boxes = m1_m2_combined + method3_boxes

        # Final aggressive deduplication: remove boxes within tight range of each other
        final_dedup = []
        for box in final_boxes:
            is_dup = False
            for existing in final_dedup:
                if (abs(existing['y'] - box['y']) <= 3 and
                    abs(existing['x'] - box['x']) <= 4 and
                    abs(existing['w'] - box['w']) < 50):
                    is_dup = True
                    break
            if not is_dup:
                final_dedup.append(box)

        final_boxes = final_dedup

        # Tighten box boundaries to fit only dark content by finding the densest horizontal band
        tightened_boxes = []
        for box in final_boxes:
            x, y, w, h = box['x'], box['y'], box['w'], box['h']

            interior_y1 = max(0, y)
            interior_y2 = min(gray.shape[0], y + h)
            interior_x1 = max(0, x)
            interior_x2 = min(gray.shape[1], x + w)

            if interior_y2 > interior_y1 and interior_x2 > interior_x1:
                interior = gray[interior_y1:interior_y2, interior_x1:interior_x2]
                black_mask = interior < 110

                black_per_row = np.sum(black_mask, axis=1)
                black_per_col = np.sum(black_mask, axis=0)

                total_cols = interior.shape[1]
                total_rows = interior.shape[0]

                # Find the densest horizontal band (core redaction bar, excluding text overlap)
                density_threshold = max(1, int(total_cols * 0.35))
                rows_with_content = black_per_row >= density_threshold
                
                if np.any(rows_with_content):
                    content_indices = np.where(rows_with_content)[0]
                    
                    # Find the densest row (peak of black pixels)
                    if len(content_indices) > 0:
                        peak_row_local = content_indices[np.argmax(black_per_row[content_indices])]
                        peak_density = black_per_row[peak_row_local]
                        
                        # Expand from peak to find solid bar boundaries, excluding sparse text overlay
                        # Use strict threshold (70%) to exclude punctuation marks but include full redaction
                        core_threshold = peak_density * 0.70
                        
                        # Find upper boundary: scan up from peak
                        core_y_start = peak_row_local
                        for i in range(peak_row_local - 1, -1, -1):
                            if black_per_row[i] >= core_threshold:
                                core_y_start = i
                            else:
                                break
                        
                        # Find lower boundary: scan down from peak
                        core_y_end = peak_row_local
                        for i in range(peak_row_local + 1, len(black_per_row)):
                            if black_per_row[i] >= core_threshold:
                                core_y_end = i
                            else:
                                break
                        
                        # Ensure minimum height and bounds validity
                        if core_y_end < core_y_start:
                            core_y_end = core_y_start
                        
                        tight_y = interior_y1 + core_y_start
                        tight_y2 = interior_y1 + core_y_end + 1
                        
                        # For columns, use moderate threshold (45% of rows) to avoid cutting off box edges
                        cols_with_content = black_per_col >= max(1, int(total_rows * 0.45))
                        if np.any(cols_with_content):
                            col_indices = np.where(cols_with_content)[0]
                            tight_x = interior_x1 + col_indices[0]
                            tight_x2 = interior_x1 + col_indices[-1] + 1
                        else:
                            tight_x = interior_x1
                            tight_x2 = interior_x2
                        
                        tight_w = tight_x2 - tight_x
                        tight_h = tight_y2 - tight_y

                        if tight_w >= 8 and tight_h >= 8:
                            tightened_boxes.append({
                                'x': tight_x, 'y': tight_y, 'w': tight_w, 'h': tight_h, 'method': box.get('method', 1)
                            })
                        else:
                            tightened_boxes.append(box)
                    else:
                        tightened_boxes.append(box)
                else:
                    tightened_boxes.append(box)
            else:
                tightened_boxes.append(box)

        final_boxes = tightened_boxes

        # Convert to PDF coordinates
        for box in final_boxes:
            x, y, w, h = box['x'], box['y'], box['w'], box['h']
            x_pt = x / scale
            y_pt = y / scale
            w_pt = w / scale
            h_pt = h / scale

            line_num = int(y_pt / 12)
            boxes_per_page[page_index] += 1
            box_num = boxes_per_page[page_index]
            uuid = f"{doc_number}-{page_index + 1}-{box_num}"

            boxes.append({
                'uuid': uuid,
                'page_index': page_index,
                'line_number': line_num,
                'x_pt': x_pt,
                'y_pt': y_pt,
                'width_pt': w_pt,
                'height_pt': h_pt,
                'width_pix': w,
                'height_pix': h
            })

            if visualize:
                color = (100, 100, 255)
                cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
                text_y = max(y - 10, 25)
                cv2.putText(img, uuid, (x, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 1)

        if visualize and boxes_per_page[page_index] > 0 and output_dir:
            os.makedirs(output_dir, exist_ok=True)
            output_path = os.path.join(output_dir, f"redaction_boxes_page_{page_index + 1}.png")
            plt.figure(figsize=(12, 16))
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            plt.title(f"Page {page_index + 1}: Detected {boxes_per_page[page_index]} redaction boxes")
            plt.axis("off")
            plt.savefig(output_path, bbox_inches='tight', dpi=150)
            plt.close()
            print(f"Saved visualization: {output_path}")

    return boxes

In [10]:
# Get text from PDF
def get_text(pdf):
    text = ""
    for page in pdf:
        text += page.get_text()
    return text

In [11]:
def extract_entities(text):
     entity_recognition = spacy.load("en_core_web_sm", disable=["tagger", "parser", "textcat"])
     doc = entity_recognition(text)
     entities = [(ent.text, ent.label_) for ent in doc.ents]
     return entities

In [12]:
def _trim_alpha(alpha):
    """Trim empty space around a glyph alpha channel."""
    if alpha.size == 0:
        return alpha
    
    # Find non-zero pixels
    coords = cv2.findNonZero(alpha)
    if coords is None:
        return alpha
    
    # Get bounding rectangle
    x, y, w, h = cv2.boundingRect(coords)
    return alpha[y:y+h, x:x+w]


def _calculate_pixel_intensity(glyph_alpha):
    """
    Calculate mean pixel intensity of foreground pixels in glyph.
    
    Args:
        glyph_alpha: binary glyph image (0-255)
    
    Returns:
        float: normalized intensity (0-1) where 1 is fully opaque/dark
    """
    if glyph_alpha.size == 0:
        return 0.0
    
    foreground_pixels = glyph_alpha[glyph_alpha > 0]
    if foreground_pixels.size == 0:
        return 0.0
    
    # Normalize to 0-1 range (255 = fully dark)
    mean_intensity = float(np.mean(foreground_pixels)) / 255.0
    return mean_intensity


def _score_glyph_quality(glyph_alpha):
    """
    Score glyph quality (0-100, higher is better).
    Penalizes fragmented/noisy glyphs, rewards solid/clean ones.
    """
    if glyph_alpha.size == 0:
        return 0.0
    
    # Solidity: ratio of foreground pixels to bounding box
    foreground = np.count_nonzero(glyph_alpha)
    h, w = glyph_alpha.shape
    bbox_area = h * w
    solidity = foreground / max(1, float(bbox_area))
    
    # Count connected components (fewer = less fragmented)
    _, labels = cv2.connectedComponents(glyph_alpha)
    num_components = labels.max()
    
    # Aspect ratio penalty (too stretched = poor)
    aspect_ratio = w / max(1, float(h))
    aspect_penalty = 1.0 if 0.3 < aspect_ratio < 3.0 else 0.5
    
    # Size penalty (very tiny = poor)
    size_penalty = 1.0 if glyph_alpha.size >= 10 else 0.3
    
    # Score: solidity (60%) + component count penalty (20%) + aspect + size
    score = (
        solidity * 60 +
        max(0, 20 - num_components * 2) +
        aspect_penalty * 10 +
        size_penalty * 10
    )
    
    return min(100.0, max(0.0, score))

In [13]:
def build_glyph_bank(
    pdf_path,
    target_chars,
    dpi=300,
    max_pages=20,
    min_area=3,
    per_char_target_samples=3,
    per_char_max_keep=8,
    quality_threshold=45.0,
    verbose=True,
):
    """
    Build a bank of character glyphs extracted from PDF(s) with early stopping.
    
    Strategy:
    - Keep only the best N samples per character (bounded memory/work).
    - Track how many high-quality samples each character has.
    - Stop scanning as soon as all target characters reach the quality target.
    
    Args:
        pdf_path: path to PDF file or list of PDF paths
        target_chars: string/iterable of characters to extract
        dpi: resolution for rendering
        max_pages: max pages to process per PDF (None = all pages, but still early-stops)
        min_area: minimum glyph area in pixels
        per_char_target_samples: number of high-quality examples needed per character
        per_char_max_keep: max candidates kept per character
        quality_threshold: minimum quality score to count toward target
        verbose: if True, print progress information
    
    Returns:
        tuple: (glyph_bank dict, list of missing chars, list of found chars, char_samples)
    """
    import time
    start_time = time.time()
    
    if per_char_target_samples < 1:
        per_char_target_samples = 1
    if per_char_max_keep < per_char_target_samples:
        per_char_max_keep = per_char_target_samples
    
    # Preserve order while removing duplicates
    target_chars = list(dict.fromkeys(target_chars))
    target_set = set(target_chars)
    
    # Handle both single PDF path and list of PDF paths
    if isinstance(pdf_path, str):
        pdf_paths = [pdf_path]
    else:
        pdf_paths = list(pdf_path)
    
    scale = dpi / 96.0
    glyph_bank = {}
    char_samples = {ch: [] for ch in target_chars}
    good_counts = {ch: 0 for ch in target_chars}
    total_glyphs_extracted = 0
    
    def all_targets_met():
        return all(good_counts[ch] >= per_char_target_samples for ch in target_chars)
    
    def add_candidate(ch, glyph_alpha, bbox):
        """Insert candidate into bounded top-N list for a character."""
        nonlocal total_glyphs_extracted
    
        quality = _score_glyph_quality(glyph_alpha)
        intensity = _calculate_pixel_intensity(glyph_alpha)
        candidate = {
            "alpha": glyph_alpha,
            "bbox": bbox,
            "baseline_ratio": 0.75 if ch.islower() else 0.80,
            "quality": quality,
            "intensity": intensity,
        }
    
        samples = char_samples[ch]
        
        if len(samples) < per_char_max_keep:
            samples.append(candidate)
            total_glyphs_extracted += 1
        else:
            # Replace worst sample only if this one is better
            worst_idx = min(range(len(samples)), key=lambda i: samples[i]["quality"])
            if quality > samples[worst_idx]["quality"]:
                samples[worst_idx] = candidate
                total_glyphs_extracted += 1
            else:
                return
    
        # Keep best samples first
        samples.sort(key=lambda s: s["quality"], reverse=True)
        good_counts[ch] = sum(1 for s in samples if s["quality"] >= quality_threshold)
    
    stop_all = False
    
    # Process each PDF
    for pdf_idx, pdf_file in enumerate(pdf_paths, 1):
        if stop_all:
            break
    
        if verbose:
            print(f"\n[{pdf_idx}/{len(pdf_paths)}] Processing: {pdf_file}")
    
        doc = fitz.open(pdf_file)
        num_pages = min(max_pages, len(doc)) if max_pages else len(doc)
    
        if verbose:
            print(f"  Pages to process (max): {num_pages}/{len(doc)}")
    
        for page_index in range(num_pages):
            if stop_all:
                break
    
            if verbose and (page_index + 1) % 5 == 0:
                met = sum(1 for ch in target_chars if good_counts[ch] >= per_char_target_samples)
                print(f"  Progress: {page_index + 1}/{num_pages} pages | chars complete: {met}/{len(target_chars)}")
    
            page = doc[page_index]
    
            # Get character-level text information
            raw = page.get_text("rawdict")
    
            # Render page as image
            mat = fitz.Matrix(scale, scale)
            pix = page.get_pixmap(matrix=mat)
            img_array = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
    
            if pix.n == 4:
                img_gray = cv2.cvtColor(img_array, cv2.COLOR_BGRA2GRAY)
            elif pix.n == 3:
                img_gray = cv2.cvtColor(img_array[:, :, :3], cv2.COLOR_RGB2GRAY)
            else:
                img_gray = img_array[:, :, 0] if pix.n == 1 else img_array
    
            # Extract characters from text
            for block in raw.get("blocks", []):
                if block.get("type") != 0:
                    continue
    
                for line in block.get("lines", []):
                    for span in line.get("spans", []):
                        for char_info in span.get("chars", []):
                            ch = char_info.get("c")
                            bbox = char_info.get("bbox")
    
                            if ch not in target_set or bbox is None:
                                continue
    
                            # Skip chars that already have enough good samples
                            if good_counts[ch] >= per_char_target_samples:
                                continue
    
                            # Convert bbox to pixels
                            x0, y0, x1, y1 = bbox
                            x0_px = int(x0 * scale)
                            y0_px = int(y0 * scale)
                            x1_px = int(x1 * scale)
                            y1_px = int(y1 * scale)
    
                            # Clip to image bounds
                            x0_px = max(0, x0_px)
                            y0_px = max(0, y0_px)
                            x1_px = min(img_gray.shape[1], x1_px)
                            y1_px = min(img_gray.shape[0], y1_px)
    
                            if x1_px <= x0_px or y1_px <= y0_px:
                                continue
    
                            # Extract glyph
                            glyph = img_gray[y0_px:y1_px, x0_px:x1_px]
                            if glyph.size < min_area:
                                continue
    
                            # Invert and threshold
                            glyph_inv = 255 - glyph
                            _, glyph_thresh = cv2.threshold(glyph_inv, 50, 255, cv2.THRESH_BINARY)
    
                            # Trim whitespace
                            glyph_trimmed = _trim_alpha(glyph_thresh)
                            if glyph_trimmed.size < min_area:
                                continue
    
                            add_candidate(ch, glyph_trimmed, bbox)
    
            # Global early stop after each page
            if all_targets_met():
                stop_all = True
                if verbose:
                    print(f"  Early stop: target reached on page {page_index + 1}")
                break
    
        doc.close()
        if verbose:
            print(f"  Completed: {page_index + 1} page(s) processed")
    
    if verbose:
        print("\nSelecting final glyphs...")
    
    # Select final glyph for each character
    for ch in target_chars:
        samples = char_samples[ch]
        if not samples:
            continue
    
        # Prefer high-quality samples first
        quality_filtered = [s for s in samples if s["quality"] >= quality_threshold]
        candidates = quality_filtered if quality_filtered else samples
    
        # Pick from top-quality candidates, tie-break by size closeness to median
        best_quality = max(s["quality"] for s in candidates)
        top_candidates = [s for s in candidates if s["quality"] >= (best_quality - 1.0)]
        median_size = float(np.median([s["alpha"].size for s in top_candidates]))
        best = min(top_candidates, key=lambda s: abs(float(s["alpha"].size) - median_size))
    
        glyph_bank[ch] = {
            "char": ch,
            "alpha": best["alpha"],
            "intensity": float(best.get("intensity", 0.5)),
            "source": "pdf",
        }
    
    found_chars = list(glyph_bank.keys())
    missing_chars = [ch for ch in target_chars if ch not in glyph_bank]
    elapsed = time.time() - start_time
    
    if verbose:
        met = sum(1 for ch in target_chars if good_counts[ch] >= per_char_target_samples)
        print("\n✓ Build complete!")
        print(f"  Found: {len(found_chars)}/{len(target_chars)} characters")
        print(f"  Missing: {len(missing_chars)} characters")
        print(f"  Chars meeting quality target: {met}/{len(target_chars)}")
        print(f"  Total candidate updates kept: {total_glyphs_extracted}")
        print(f"  Time elapsed: {elapsed:.1f}s")
    
    return glyph_bank, missing_chars, found_chars, char_samples

In [14]:
def match_names_to_redactions(boxes, name_widths, glyph_bank, pdf, dpi=300):
    """
    Match names to redaction boxes based on width similarity.
    
    Args:
        boxes: list of redaction boxes (page, line, x, y, w, h)
        name_widths: list of dicts with 'name' key
        glyph_bank: dictionary of character glyphs
        pdf: fitz PDF document
        dpi: resolution
    
    Returns:
        dict: mapping of box_key to match info {name, name_width_px, box_width_px, width_diff_px}
    """
    from overlay_function import _build_page_spacing_models
    
    scale = dpi / 96.0
    matched_pairs = {}
    
    # Build spacing models for all pages
    spacing_cache = {}
    for page_idx in range(len(pdf)):
        page = pdf[page_idx]
        spacing_cache[page_idx] = _build_page_spacing_models(page, scale)
    
    # Calculate width for each name using glyphs
    def calc_name_width(name, spacing_model, y_center_pt):
        line_pair_map = {}
        line_space_px = spacing_model["page_space_px"]
        
        if spacing_model["line_models"] and y_center_pt is not None:
            nearest = min(spacing_model["line_models"], 
                         key=lambda lm: abs(lm["y_center"] - float(y_center_pt)))
            line_pair_map = nearest.get("pair_map", {})
            if nearest.get("space_px") is not None:
                line_space_px = nearest.get("space_px")
        
        total_width = 0.0
        prev_char = None
        
        for ch in name:
            if ch == " ":
                total_width += float(line_space_px)
                prev_char = None
                continue
            
            entry = glyph_bank.get(ch) or glyph_bank.get(ch.swapcase())
            if entry is None or "alpha" not in entry:
                prev_char = ch
                continue
            
            glyph_w = entry["alpha"].shape[1]
            
            if prev_char is not None:
                # Simple kerning lookup
                pair = f"{prev_char}{ch}"
                kern = line_pair_map.get(pair, 1.0)
                if kern == 0.0:
                    kern = 1.0
                total_width += kern
            
            total_width += float(glyph_w)
            prev_char = ch
        
        return total_width if total_width > 0 else None
    
    # Match each box to closest name by width
    for box in boxes:
        # Extract values from box dictionary
        page_idx = box['page_index']
        line_num = box['line_number']
        x_pt = box['x_pt']
        y_pt = box['y_pt']
        w_pt = box['width_pt']
        h_pt = box['height_pt']
        uuid = box['uuid']
        
        box_width_px = w_pt * scale
        y_center_pt = y_pt + (h_pt / 2.0)
        
        spacing_model = spacing_cache.get(int(page_idx))
        if spacing_model is None:
            continue
        
        best_match = None
        best_diff = float('inf')
        
        for name_info in name_widths:
            name = name_info['name']
            name_width_px = calc_name_width(name, spacing_model, y_center_pt)
            
            if name_width_px is None:
                continue
            
            width_diff = abs(box_width_px - name_width_px)
            
            # Accept if within 20% of box width
            if width_diff < box_width_px * 0.20 and width_diff < best_diff:
                best_diff = width_diff
                best_match = {
                    "name": name,
                    "name_width_px": name_width_px,
                    "box_width_px": box_width_px,
                    "width_diff_px": width_diff,
                    "uuid": uuid
                }
        
        if best_match:
            box_key = (page_idx, line_num, x_pt, y_pt, w_pt, h_pt)
            matched_pairs[box_key] = best_match
    
    return matched_pairs

In [ ]:
boxes = extract_redaction_boxes(DOC, black_threshold=100, min_width=5, visualize=True, output_dir="overlays")

In [16]:
def get_line_font_info(pdf):
    """
    Returns a list of:
    {
        page,
        line_number,
        font_size,
        font_name
    }
    """

    results = []

    for page_index, page in enumerate(pdf):

        text = page.get_text("dict")
        line_number = 0

        for block in text["blocks"]:
            if "lines" not in block:
                continue

            for line in block["lines"]:
                line_number += 1

                spans = line["spans"]
                if not spans:
                    continue

                # Most lines use one dominant font/size,
                # but spans can differ within a line.
                # We'll take the most common combination.

                font_pairs = [
                    (round(span["size"], 2), span["font"])
                    for span in spans
                ]

                most_common = Counter(font_pairs).most_common(1)[0][0]

                results.append({
                    "page": page_index,
                    "line_number": line_number,
                    "font_size": most_common[0],
                    "font_name": most_common[1]
                })

    return results

In [17]:
print(get_line_font_info(DOC))

[{'page': 0, 'line_number': 1, 'font_size': 12.32, 'font_name': 'Times-Roman'}, {'page': 0, 'line_number': 2, 'font_size': 12.1, 'font_name': 'Times-Roman'}, {'page': 0, 'line_number': 3, 'font_size': 12.63, 'font_name': 'Times-Roman'}, {'page': 0, 'line_number': 4, 'font_size': 12.11, 'font_name': 'Times-Roman'}, {'page': 0, 'line_number': 5, 'font_size': 12.55, 'font_name': 'Times-Roman'}, {'page': 0, 'line_number': 6, 'font_size': 12.15, 'font_name': 'Times-Roman'}, {'page': 0, 'line_number': 7, 'font_size': 11.87, 'font_name': 'Times-Roman'}, {'page': 0, 'line_number': 8, 'font_size': 12.02, 'font_name': 'Times-Roman'}, {'page': 0, 'line_number': 9, 'font_size': 12.01, 'font_name': 'Times-Roman'}, {'page': 0, 'line_number': 10, 'font_size': 12.03, 'font_name': 'Times-Roman'}, {'page': 0, 'line_number': 11, 'font_size': 11.89, 'font_name': 'Times-Roman'}, {'page': 0, 'line_number': 12, 'font_size': 11.99, 'font_name': 'Times-Roman'}, {'page': 0, 'line_number': 13, 'font_size': 11.66

In [ ]:
boxes = extract_redaction_boxes(DOC, black_threshold=240, min_width=5, visualize=True, output_dir="overlays")

In [19]:

def estimate_redaction_char_count(redaction_width, min_char_width, max_char_width):
    """
    Estimate minimum and maximum character count that could fit in a redaction box.
    
    Uses character width metrics to determine bounds on how many characters could
    be hidden under a redaction.
    
    Args:
        redaction_width: width of the redaction box in pixels
        min_char_width: minimum character width in pixels (narrowest letter like 'i')
        max_char_width: maximum character width in pixels (widest letter like 'W')
    
    Returns:
        dict with:
            min_chars: minimum possible characters (using widest char width)
            max_chars: maximum possible characters (using narrowest char width)
            range: tuple of (min_chars, max_chars)
    """
    # Minimum characters: if all chars are the widest
    min_chars = int(redaction_width / max_char_width)
    
    # Maximum characters: if all chars are the narrowest
    max_chars = int(redaction_width / min_char_width)
    
    return {
        "redaction_width": redaction_width,
        "min_char_width": min_char_width,
        "max_char_width": max_char_width,
        "min_chars": min_chars,
        "max_chars": max_chars,
        "range": (min_chars, max_chars)
    }

In [20]:
# Synthetic glyph generation functions

def _render_synthetic_char(ch, target_h, target_w):
    """Render a character using OpenCV's built-in font as fallback."""
    h = max(28, target_h)
    w = max(20, target_w)
    canvas = np.zeros((h * 2, w * 2), dtype=np.uint8)
    font = cv2.FONT_HERSHEY_SIMPLEX
    thickness = 2 if ch.isupper() else 1
    scale = 0.9 if ch.islower() else 1.0

    (tw, th), _ = cv2.getTextSize(ch, font, scale, thickness)
    x = max(2, (canvas.shape[1] - tw) // 2)
    y = max(th + 2, (canvas.shape[0] + th) // 2)
    cv2.putText(canvas, ch, (x, y), font, scale, 255, thickness, cv2.LINE_AA)

    return _trim_alpha(canvas)


def _promote_case(alpha, to_upper):
    """
    Generate uppercase from lowercase glyph or vice versa by resizing.
    Uppercase glyphs are scaled larger and thickened slightly.
    """
    h, w = alpha.shape[:2]
    if to_upper:
        nh = max(1, int(round(h * 1.25)))
        nw = max(1, int(round(w * 1.15)))
    else:
        nh = max(1, int(round(h * 0.82)))
        nw = max(1, int(round(w * 0.90)))
    out = cv2.resize(alpha, (nw, nh), interpolation=cv2.INTER_LINEAR)
    if to_upper:
        out = cv2.dilate(out, np.ones((2, 2), dtype=np.uint8), iterations=1)
    return _trim_alpha(out)


def apply_synthetic_fallback(glyph_bank, target_chars):
    """
    Fill in missing characters using synthetic generation strategies:
    1. Case conversion (e.g., derive 'A' from 'a')
    2. Shape-based modifications (e.g., derive 'Q' from 'O' with added tail)
    3. OpenCV font rendering as final fallback
    """
    added = []

    def estimate_dims():
        hs, ws = [], []
        for v in glyph_bank.values():
            a = v["alpha"]
            hs.append(a.shape[0])
            ws.append(a.shape[1])
        if not hs:
            return 30, 20
        return int(np.median(hs)), int(np.median(ws))

    med_h, med_w = estimate_dims()

    # 1) case-derived fallback
    for ch in target_chars:
        if ch in glyph_bank:
            continue
        pair = ch.swapcase()
        if pair in glyph_bank:
            alpha = _promote_case(glyph_bank[pair]["alpha"], to_upper=ch.isupper())
            intensity = _calculate_pixel_intensity(alpha)
            glyph_bank[ch] = {
                "char": ch,
                "alpha": alpha,
                "score": 9.0,
                "baseline_ratio": glyph_bank[pair].get("baseline_ratio", 0.80),
                "intensity": intensity,
                "source": f"synthetic_case_from_{pair}",
            }
            added.append(ch)

    # 2) shape-derived fallback for common misses
    shape_map = {
        "q": "p",
        "Q": "O",
        "P": "p",
        "R": "P",
        "U": "u",
        "V": "v",
        "X": "x",
        "Z": "z",
        "G": "C",
        "H": "N",
        "K": "k",
    }

    for ch, base in shape_map.items():
        if ch in glyph_bank or base not in glyph_bank:
            continue

        alpha = glyph_bank[base]["alpha"].copy()

        if ch == "q" and "p" in glyph_bank:
            alpha = cv2.flip(glyph_bank["p"]["alpha"], 1)
        elif ch == "Q" and "O" in glyph_bank:
            alpha = glyph_bank["O"]["alpha"].copy()
            h, w = alpha.shape
            cv2.line(alpha, (int(0.62 * w), int(0.62 * h)), (w - 1, h - 1), 255, 1)
        elif ch == "R" and "P" in glyph_bank:
            alpha = glyph_bank["P"]["alpha"].copy()
            h, w = alpha.shape
            cv2.line(alpha, (int(0.45 * w), int(0.55 * h)), (w - 1, h - 1), 255, 1)
        elif ch == "G" and "C" in glyph_bank:
            alpha = glyph_bank["C"]["alpha"].copy()
            h, w = alpha.shape
            cv2.line(alpha, (int(0.55 * w), int(0.55 * h)), (w - 1, int(0.55 * h)), 255, 1)

        if ch.isupper() and base.islower():
            alpha = _promote_case(alpha, to_upper=True)
        intensity = _calculate_pixel_intensity(alpha)
        glyph_bank[ch] = {
            "char": ch,
            "alpha": alpha,
            "score": 9.5,
            "baseline_ratio": glyph_bank[base].get("baseline_ratio", 0.80),
            "intensity": intensity,
            "source": f"synthetic_shape_from_{base}",
        }
        added.append(ch)

    # 3) final hard fallback: synthetic draw for any still-missing glyphs
    for ch in target_chars:
        if ch in glyph_bank:
            continue
        alpha = _render_synthetic_char(ch, med_h, med_w)
        glyph_bank[ch] = {
            "char": ch,
            "alpha": alpha,
            "score": 10.0,
            "baseline_ratio": (0.78 if ch.isupper() else 0.82),
            "source": "synthetic_draw",
        }
        added.append(ch)

    missing_after = [ch for ch in target_chars if ch not in glyph_bank]
    return glyph_bank, sorted(set(added)), missing_after


In [21]:
# Glyph bank save and visualization functions

def save_glyph_bank(glyph_bank, out_dir):
    """Save each glyph as a separate PNG file."""
    os.makedirs(out_dir, exist_ok=True)

    for ch, meta in glyph_bank.items():
        alpha = meta["alpha"]
        filename = f"{ord(ch):03d}_{ch}.png"
        cv2.imwrite(os.path.join(out_dir, filename), alpha)


def analyze_glyph_samples(char_samples, target_chars, quality_threshold=40.0):
    """
    Analyze quality and availability of glyph samples.
    Helps identify which characters have poor candidates.
    
    Returns dict with analysis for each character.
    """
    analysis = {}
    
    for ch in sorted(target_chars):
        samples = char_samples.get(ch, [])
        
        if not samples:
            analysis[ch] = {
                'count': 0,
                'avg_quality': 0,
                'best_quality': 0,
                'status': 'MISSING'
            }
            continue
        
        # Score all samples
        scores = [_score_glyph_quality(s["alpha"]) for s in samples]
        avg_quality = float(np.mean(scores))
        best_quality = max(scores)
        high_quality_count = sum(1 for s in scores if s >= quality_threshold)
        
        status = 'GOOD' if best_quality >= 60 else 'FAIR' if best_quality >= 40 else 'POOR'
        
        analysis[ch] = {
            'count': len(samples),
            'avg_quality': avg_quality,
            'best_quality': best_quality,
            'high_quality_count': high_quality_count,
            'status': status
        }
    
    return analysis


def select_best_glyph_manually(char, char_samples, show_count=5):
    """
    Display top N glyph candidates for a character and let user pick the best.
    Useful for improving specific characters like 'M'.
    """
    samples = char_samples.get(char, [])
    if not samples:
        print(f"No samples found for '{char}'")
        return None
    
    # Score and sort by quality
    scored = []
    for idx, sample in enumerate(samples):
        quality = _score_glyph_quality(sample["alpha"])
        scored.append((quality, idx, sample))
    
    scored.sort(reverse=True, key=lambda x: x[0])
    
    # Show top candidates
    print(f"\nTop {min(show_count, len(scored))} candidates for '{char}':")
    print(f"{'Rank':<6} {'Quality':<10} {'Size':<12} {'Solidity':<10}")
    print("-" * 40)
    
    for rank, (quality, idx, sample) in enumerate(scored[:show_count], 1):
        alpha = sample["alpha"]
        h, w = alpha.shape
        fg = np.count_nonzero(alpha)
        solidity = fg / (h * w) if h * w > 0 else 0
        
        print(f"{rank:<6} {quality:<10.1f} {h}×{w:<9} {solidity:<10.2f}")
    
    return scored[:show_count]


def save_glyph_bank(glyph_bank, out_dir):
    """Save each glyph as a separate PNG file."""
    os.makedirs(out_dir, exist_ok=True)

    for ch, meta in glyph_bank.items():
        alpha = meta["alpha"]
        filename = f"{ord(ch):03d}_{ch}.png"
        cv2.imwrite(os.path.join(out_dir, filename), alpha)


def save_glyph_atlas(glyph_bank, out_path, columns=13, cell_w=60, cell_h=72):
    """
    Create a visual atlas showing all glyphs in a grid.
    Synthetic glyphs are marked with an asterisk.
    """
    keys = sorted(glyph_bank.keys(), key=lambda c: (c.islower(), c))
    rows = int(np.ceil(len(keys) / columns))

    atlas_h = rows * cell_h
    atlas_w = columns * cell_w
    atlas = np.full((atlas_h, atlas_w, 3), 255, dtype=np.uint8)

    for idx, ch in enumerate(keys):
        r = idx // columns
        c = idx % columns
        x0 = c * cell_w
        y0 = r * cell_h

        alpha = glyph_bank[ch]["alpha"]
        gh, gw = alpha.shape[:2]

        gx = x0 + max(0, (cell_w - gw) // 2)
        gy = y0 + max(10, (cell_h - gh) // 2)

        gx1 = min(atlas_w, gx + gw)
        gy1 = min(atlas_h, gy + gh)
        a = alpha[: gy1 - gy, : gx1 - gx]

        roi = atlas[gy:gy1, gx:gx1]
        mask = (a > 0).astype(np.uint8)
        roi[mask > 0] = (0, 200, 0)

        src = glyph_bank[ch].get("source", "pdf")
        label = ch if src == "pdf" else f"{ch}*"

        cv2.putText(
            atlas,
            label,
            (x0 + 4, y0 + cell_h - 6),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.42,
            (20, 20, 20),
            1,
            cv2.LINE_AA,
        )

    cv2.imwrite(out_path, atlas)

In [22]:
# Analyze glyph quality and identify problematic characters
data_ready = True

if 'char_samples' not in globals() or char_samples is None:
    if 'glyph_bank' in globals() and glyph_bank:
        print("char_samples not found; building a minimal view from glyph_bank for analysis.")
        char_samples = {
            ch: [{"alpha": meta["alpha"]}]
            for ch, meta in glyph_bank.items()
            if isinstance(meta, dict) and "alpha" in meta
        }
    else:
        print("char_samples is not defined. Run Cell 17 first (glyph bank build), then rerun this cell.")
        data_ready = False

if data_ready:
    if 'target_chars' not in globals() or target_chars is None:
        target_chars = sorted(char_samples.keys())
    elif isinstance(target_chars, str):
        target_chars = list(target_chars)

    analysis = analyze_glyph_samples(char_samples, target_chars, quality_threshold=40.0)

    # Show characters with poor quality
    poor_glyphs = {ch: a for ch, a in analysis.items() if a['status'] == 'POOR'}
    fair_glyphs = {ch: a for ch, a in analysis.items() if a['status'] == 'FAIR'}

    print("POOR QUALITY GLYPHS (status='POOR'):")
    for ch in sorted(poor_glyphs.keys()):
        a = poor_glyphs[ch]
        print(f"  '{ch}': {a['count']} samples, best quality: {a['best_quality']:.1f}")

    print("\nFAIR QUALITY GLYPHS (status='FAIR'):")
    for ch in sorted(fair_glyphs.keys()):
        a = fair_glyphs[ch]
        print(f"  '{ch}': {a['count']} samples, best quality: {a['best_quality']:.1f}")

    # Show how many characters have good quality
    good_glyphs = {ch: a for ch, a in analysis.items() if a['status'] == 'GOOD'}
    print(f"\nGOOD QUALITY: {len(good_glyphs)} characters")
    print(f"FAIR QUALITY: {len(fair_glyphs)} characters")
    print(f"POOR QUALITY: {len(poor_glyphs)} characters")

char_samples is not defined. Run Cell 17 first (glyph bank build), then rerun this cell.


In [ ]:
# Build the glyph bank from multiple PDFs and apply synthetic fallbacks
target_chars = string.ascii_lowercase + string.ascii_uppercase

# List of PDFs to extract glyphs from for comprehensive character coverage
pdf_paths = [
    os.path.join(PDF_FOLDER, 'EFTA00037360.pdf'),
    os.path.join(PDF_FOLDER, 'EFTA00037361.pdf'),
    os.path.join(PDF_FOLDER, 'EFTA00037362.pdf'),
    os.path.join(PDF_FOLDER, 'EFTA00037363.pdf'),
    os.path.join(PDF_FOLDER, 'EFTA00037364.pdf'),
    os.path.join(PDF_FOLDER, 'EFTA00037365.pdf'),
]

# Extract glyphs with bounded sampling + automatic early stop
glyph_bank, missing_chars, _, char_samples = build_glyph_bank(
    pdf_path=pdf_paths,
    target_chars=target_chars,
    dpi=DPI,
    max_pages=None,                # Early-stop controls runtime
    min_area=3,
    per_char_target_samples=4,     # slightly higher sample target for better quality
    per_char_max_keep=10,          # keep a few more top candidates per glyph
    quality_threshold=45.0,
    verbose=True
 )

# Fill in missing glyphs synthetically
glyph_bank, synthetic_added, missing_after = apply_synthetic_fallback(glyph_bank, target_chars)

print(f"\nSynthetic fallbacks added: {len(synthetic_added)} characters")
print(f"Still missing: {len(missing_after)} characters")

# Optionally save to disk (comment out if want to skip file I/O)
bank_dir = os.path.join(os.getcwd(), "glyph_bank")
atlas_path = os.path.join(os.getcwd(), "glyph_bank_atlas.png")
save_glyph_bank(glyph_bank, bank_dir)

save_glyph_atlas(glyph_bank, atlas_path, columns=13, cell_w=60, cell_h=72)
print("✓ Glyphs saved to disk")

In [24]:
# To improve a specific character (e.g., 'M'), examine top candidates:
# 
# Top candidates for 'M':
candidates_M = select_best_glyph_manually('M', char_samples, show_count=10)

# To replace 'M' with a better candidate, do:
# if candidates_M:
#     best_quality, best_idx, best_sample = candidates_M[0]  # Use top candidate
#     glyph_bank['M']['alpha'] = best_sample['alpha']
#     print("✓ Updated 'M' with better candidate")
#     save_glyph_atlas(glyph_bank, atlas_path, columns=13, cell_w=60, cell_h=72)


Top 4 candidates for 'M':
Rank   Quality    Size         Solidity  
----------------------------------------
1      73.2       23×25        0.59      
2      69.6       26×31        0.53      
3      67.3       26×30        0.49      
4      65.0       26×31        0.45      


In [25]:
# Improve problematic glyphs: K, Q, X, Z, W (uppercase) and f, k (lowercase)
problem_chars = ['K', 'Q', 'X', 'Z', 'W', 'f', 'k']

print("Analyzing problem glyphs for improvement...\n")

for ch in problem_chars:
    samples = char_samples.get(ch, [])
    if not samples:
        print(f"'{ch}': no samples found")
        continue
    
    # Score all samples
    scored = []
    for idx, sample in enumerate(samples):
        quality = _score_glyph_quality(sample["alpha"])
        scored.append((quality, idx, sample))
    
    scored.sort(reverse=True, key=lambda x: x[0])
    
    best_quality, best_idx, best_sample = scored[0]
    current_quality = _score_glyph_quality(glyph_bank[ch]["alpha"])
    
    if best_quality > current_quality + 1.0:
        # Replace with better candidate
        glyph_bank[ch]["alpha"] = best_sample["alpha"]
        print(f"✓ '{ch}': improved from {current_quality:.1f} → {best_quality:.1f}")
    else:
        print(f"  '{ch}': current {current_quality:.1f} vs best available {best_quality:.1f} (no improvement needed)")

# Regenerate atlas with improved glyphs
save_glyph_atlas(glyph_bank, atlas_path, columns=13, cell_w=60, cell_h=72)
save_glyph_bank(glyph_bank, bank_dir)
print("\n✓ Glyph bank updated and atlas regenerated")

Analyzing problem glyphs for improvement...

'K': no samples found
'Q': no samples found
'X': no samples found
'Z': no samples found
  'W': current 64.4 vs best available 64.8 (no improvement needed)
  'f': current 70.3 vs best available 70.3 (no improvement needed)
  'k': current 67.3 vs best available 67.3 (no improvement needed)

✓ Glyph bank updated and atlas regenerated


In [ ]:
# Load the workbook
wb = openpyxl.load_workbook('Redactions Analysis.xlsx')

# Get the Names sheet
names_sheet = wb['Names']

# Read column A starting from row 3
names_list = []
for row in range(3, names_sheet.max_row + 1):
    cell_value = names_sheet[f'A{row}'].value
    if cell_value is not None:  # Only add non-empty cells
        names_list.append(cell_value)

In [27]:
# Convert names_list to the format expected by match_names_to_redactions
name_widths_for_matching = [{'name': name} for name in names_list]

In [ ]:
# Match names to redaction boxes
matched_pairs = match_names_to_redactions(
    boxes=boxes,
    name_widths=name_widths_for_matching,
    glyph_bank=glyph_bank,
    pdf=DOC,
    dpi=DPI
)

print(f"Matched {len(matched_pairs)} names to redaction boxes")
print(f"Sample matches: {list(matched_pairs.values())[:3]}")

In [ ]:
def _median_pair_map(pair_samples):
    """Compute median spacing for each character pair."""
    out = {}
    for key, vals in pair_samples.items():
        if vals:
            out[key] = float(np.median(vals))
    return out


def _build_page_spacing_models(page, scale_px_per_pt):
    raw = page.get_text("rawdict")
    line_models = []
    page_pair_samples = {}
    page_space_samples = []

    for block in raw.get("blocks", []):
        if block.get("type") != 0:
            continue

        for line in block.get("lines", []):
            line_chars = []
            line_font_sizes = []
            for span in line.get("spans", []):
                font_size = span.get("size", 12.0)
                line_font_sizes.append(float(font_size))
                
                for ch in span.get("chars", []):
                    c = ch.get("c", "")
                    bb = ch.get("bbox")
                    if not c or bb is None:
                        continue
                    line_chars.append((c, bb))

            if not line_chars:
                continue

            line_pair_samples = {}
            line_space_samples = []

            for c, bb in line_chars:
                if c == " ":
                    space_w_px = float(np.clip((bb[2] - bb[0]) * scale_px_per_pt, 1.0, 24.0))
                    line_space_samples.append(space_w_px)
                    page_space_samples.append(space_w_px)

            for i in range(len(line_chars) - 1):
                c1, bb1 = line_chars[i]
                c2, bb2 = line_chars[i + 1]
                if c1.isspace() or c2.isspace():
                    continue

                gap_pt = float(bb2[0] - bb1[2])
                gap_px = float(np.clip(gap_pt * scale_px_per_pt, -3.0, 8.0))
                pair = f"{c1}{c2}"

                line_pair_samples.setdefault(pair, []).append(gap_px)
                page_pair_samples.setdefault(pair, []).append(gap_px)

            line_pair_map = _median_pair_map(line_pair_samples)
            line_space_px = float(np.median(line_space_samples)) if line_space_samples else None

            if line_pair_map or line_space_px is not None:
                y0 = min(float(bb[1]) for _, bb in line_chars)
                y1 = max(float(bb[3]) for _, bb in line_chars)
                y_center = 0.5 * (y0 + y1)
                baseline_y_px = int((y0 + 0.8 * (y1 - y0)) * scale_px_per_pt)
                font_size_pt = float(np.median(line_font_sizes)) if line_font_sizes else 12.0
                text_height_px = int(font_size_pt * scale_px_per_pt)
                
                line_models.append({
                    "y_center": y_center,
                    "pair_map": line_pair_map,
                    "space_px": line_space_px,
                    "baseline_y_px": baseline_y_px,
                    "text_height_px": text_height_px,
                })

    page_pair_map = _median_pair_map(page_pair_samples)
    page_space_px = float(np.median(page_space_samples)) if page_space_samples else 6.0

    return {
        "line_models": line_models,
        "page_pair_map": page_pair_map,
        "page_space_px": page_space_px,
    }


def _lookup_pair_gap(ch_left, ch_right, line_pair_map, page_pair_map):
    """Look up kerning between two characters with case fallbacks."""
    candidates = []
    seen = set()
    for pair in [
        f"{ch_left}{ch_right}",
        f"{ch_left.lower()}{ch_right.lower()}",
        f"{ch_left.upper()}{ch_right.upper()}",
        f"{ch_left.upper()}{ch_right.lower()}",
        f"{ch_left.lower()}{ch_right.upper()}",
    ]:
        if pair not in seen:
            candidates.append(pair)
            seen.add(pair)
    
    for pair in candidates:
        if pair in line_pair_map:
            gap = float(line_pair_map[pair])
            if gap == 0.0:
                gap += 1.0
            return gap
    for pair in candidates:
        if pair in page_pair_map:
            gap = float(page_pair_map[pair])
            if gap == 0.0:
                gap += 1.0
            return gap
    
    return 1.0


def _name_width_from_glyphs_and_learned_spacing(name, glyph_bank, spacing_model, target_center_y_pt):
    """Calculate the pixel width of a name using glyphs and learned kerning."""
    line_pair_map = {}
    line_space_px = spacing_model["page_space_px"]
    
    if spacing_model["line_models"] and target_center_y_pt is not None:
        nearest = min(spacing_model["line_models"], key=lambda lm: abs(lm["y_center"] - float(target_center_y_pt)))
        line_pair_map = nearest.get("pair_map", {})
        if nearest.get("space_px") is not None:
            line_space_px = nearest.get("space_px")
    
    total_width_px = 0.0
    prev_char = None
    
    for ch in name:
        if ch == " ":
            total_width_px += float(line_space_px)
            prev_char = None
            continue
        
        entry = glyph_bank.get(ch) or glyph_bank.get(ch.swapcase())
        if entry is None or "alpha" not in entry:
            prev_char = ch
            continue
        
        glyph_alpha = entry["alpha"]
        glyph_w = glyph_alpha.shape[1]
        
        if prev_char is not None:
            kern = _lookup_pair_gap(prev_char, ch, line_pair_map, spacing_model["page_pair_map"])
            total_width_px += kern
        
        total_width_px += float(glyph_w)
        prev_char = ch
    
    return total_width_px if total_width_px > 0 else None

In [ ]:
# Debug: Check what font sizes are being extracted from the PDF
scale = DPI / 96.0

with fitz.open(pdf_path) as doc:
    page = doc[0]
    raw = page.get_text("rawdict")
    font_sizes_found = []
    
    for block in raw.get("blocks", []):
        if block.get("type") != 0:
            continue
        for line in block.get("lines", []):
            for span in line.get("spans", []):
                font_size = span.get("size", None)
                if font_size:
                    font_sizes_found.append(font_size)
    
    print(f"Font sizes found in PDF: {sorted(set(font_sizes_found))}")
    if font_sizes_found:
        print(f"Median font size: {np.median(font_sizes_found):.2f} pt")
        print(f"Scale factor (DPI/96): {scale:.4f}")
        print(f"Expected text height in pixels: {int(np.median(font_sizes_found) * scale)}")

Font sizes found in PDF: [11.648354530334473, 11.655129432678223, 11.67481517791748, 11.70489501953125, 11.714699745178223, 11.769092559814453, 11.809266090393066, 11.842034339904785, 11.856568336486816, 11.868667602539062, 11.869272232055664, 11.874711990356445, 11.881357192993164, 11.883772850036621, 11.886188507080078, 11.887999534606934, 11.892827987670898, 11.895240783691406, 11.905491828918457, 11.92115306854248, 11.926568984985352, 11.941000938415527, 11.94160270690918, 11.95061206817627, 11.952413558959961, 11.960816383361816, 11.969212532043457, 11.969812393188477, 11.97520637512207, 11.975805282592773, 11.979999542236328, 11.991973876953125, 11.993768692016602, 11.994366645812988, 11.994965553283691, 11.997956275939941, 11.999751091003418, 12.0, 12.000348091125488, 12.002141952514648, 12.003337860107422, 12.004533767700195, 12.005130767822266, 12.006326675415039, 12.00871753692627, 12.00931453704834, 12.010510444641113, 12.012900352478027, 12.013497352600098, 12.0164833068847

In [ ]:
# Debug: Check glyph heights in the bank
glyph_heights = []
for ch, entry in glyph_bank.items():
    if "alpha" in entry:
        h = entry["alpha"].shape[0]
        glyph_heights.append(h)

median_glyph_h = np.median(glyph_heights)
print(f"\nGlyph bank analysis:")
print(f"Median glyph height: {median_glyph_h:.1f} pixels")
print(f"Min glyph height: {min(glyph_heights)}")
print(f"Max glyph height: {max(glyph_heights)}")

# Expected scale factor
text_height_px = 50  # from debug above
expected_scale_factor = text_height_px / median_glyph_h
print(f"\nExpected scale factor: {expected_scale_factor:.4f}")
print(f"This would scale glyphs from {median_glyph_h:.0f}px to {text_height_px}px")


Glyph bank analysis:
Median glyph height: 25.0 pixels
Min glyph height: 16
Max glyph height: 34

Expected scale factor: 2.0000
This would scale glyphs from 25px to 50px


In [ ]:
print(matched_pairs)
print(len(matched_pairs))

{(0, 3, np.float64(108.16), 40.64, np.float64(272.32), 12.48): {'name': 'Sheikh Hamad bin Jassim bin Jaber Al Thani (HBJ)', 'name_width_px': 696.2402267456055, 'box_width_px': np.float64(851.0), 'width_diff_px': np.float64(154.75977325439453), 'uuid': 'EFTA00037366-1-1'}, (0, 12, np.float64(33.92), np.float64(153.92), np.float64(44.16), np.float64(12.48)): {'name': 'Bill Gates', 'name_width_px': 136.45062637329102, 'box_width_px': np.float64(138.0), 'width_diff_px': np.float64(1.5493736267089844), 'uuid': 'EFTA00037366-1-3'}, (0, 12, np.float64(319.68), np.float64(153.92), np.float64(31.36), np.float64(12.48)): {'name': 'Joi Ito', 'name_width_px': 86.45062637329102, 'box_width_px': np.float64(98.0), 'width_diff_px': np.float64(11.549373626708984), 'uuid': 'EFTA00037366-1-4'}, (0, 15, np.float64(149.44), 180.8, np.float64(31.36), 12.8): {'name': 'Joi Ito', 'name_width_px': 86.41927433013916, 'box_width_px': np.float64(98.0), 'width_diff_px': np.float64(11.58072566986084), 'uuid': 'EFTA0

In [ ]:
def _render_name_with_glyphs(name, glyph_bank, spacing_model, target_center_y_pt, start_x_px, start_y_px):
    """
    Render a name using glyph bank characters with learned kerning.
    All characters are aligned to a common baseline for proper vertical alignment.
    
    Returns list of (glyph_alpha, x_pos, y_pos) tuples.
    """
    line_pair_map = {}
    line_space_px = None
    
    # Find nearest line model based on y position
    if spacing_model["line_models"] and target_center_y_pt is not None:
        nearest = min(spacing_model["line_models"], key=lambda lm: abs(lm["y_center"] - float(target_center_y_pt)))
        line_pair_map = nearest.get("pair_map", {})
        line_space_px = nearest.get("space_px")
    
    if line_space_px is None:
        line_space_px = spacing_model["page_space_px"]
    
    # First pass: collect glyphs to determine alignment
    temp_glyphs = []
    current_x = float(start_x_px)
    prev_char = None
    
    for ch in name:
        if ch == " ":
            current_x += float(line_space_px)
            prev_char = None
            continue
        
        # Get glyph (try exact case, then swapcase)
        entry = glyph_bank.get(ch) or glyph_bank.get(ch.swapcase())
        if entry is None or "alpha" not in entry:
            prev_char = ch
            continue
        
        glyph_alpha = entry["alpha"]
        glyph_h, glyph_w = glyph_alpha.shape[:2]
        baseline_ratio = entry.get("baseline_ratio", 0.80)
        
        # Add kerning from previous character
        if prev_char is not None:
            kern = _lookup_pair_gap(prev_char, ch, line_pair_map, spacing_model["page_pair_map"])
            current_x += kern
        
        temp_glyphs.append((glyph_alpha, int(current_x), glyph_h, baseline_ratio))
        
        current_x += glyph_w
        prev_char = ch
    
    # Use a common baseline position for all glyphs
    common_baseline_y = start_y_px
    
    # Second pass: position all glyphs aligned to common baseline
    glyphs_to_render = []
    for glyph_alpha, glyph_x, glyph_h, baseline_ratio in temp_glyphs:
        # Position each glyph so its baseline aligns with the common baseline
        baseline_offset = baseline_ratio * glyph_h
        glyph_y = int(common_baseline_y - baseline_offset)
        glyphs_to_render.append((glyph_alpha, glyph_x, glyph_y))
    
    return glyphs_to_render


In [ ]:
# Generate overlays for all matched names
overlay_files = overlay_names_on_redaction_boxes(
    matched_pairs=matched_pairs,
    pdf_path=pdf_path,
    glyph_bank=glyph_bank,
    output_dir="overlays",
    dpi=DPI,
    visualize_each=False  # Set to True to display each image in the notebook
)

In [ ]:
# Analyze redaction boxes detected with threshold=240
print(f"Total redaction boxes detected: {len(boxes)}")
print(f"\nFirst 5 boxes:")
for i, box in enumerate(boxes[:5]):
    print(f"\nBox {i+1} - UUID: {box['uuid']}")
    print(f"  Page: {box['page_index'] + 1}, Line: {box['line_number']}")
    print(f"  Dimensions: {box['width_pix']}×{box['height_pix']} pixels")
    print(f"  Position: ({box['x_pt']:.1f}, {box['y_pt']:.1f}) points")

# Show width distribution
widths = [box['width_pix'] for box in boxes]
heights = [box['height_pix'] for box in boxes]
print(f"\n\nWidth statistics:")
print(f"  Min: {min(widths)}px, Max: {max(widths)}px, Mean: {np.mean(widths):.1f}px")
print(f"Height statistics:")
print(f"  Min: {min(heights)}px, Max: {max(heights)}px, Mean: {np.mean(heights):.1f}px")

# Display the visualization
viz_img = Image.open(os.path.join('overlays', 'redaction_boxes_page_1.png'))
plt.figure(figsize=(14, 18))
plt.imshow(viz_img)
plt.axis('off')
plt.title(f"Redaction Boxes (threshold=240) - {len([b for b in boxes if b['page_index'] == 0])} boxes on page 1")
plt.tight_layout()
plt.show()

In [ ]:
# Write ALL redaction boxes to file for inspection
output_path = 'redaction_analysis_threshold240.txt'

with open(output_path, 'w') as f:
    f.write("REDACTION BOX ANALYSIS (threshold=240) - ALL DETECTED BOXES\n")
    f.write("=" * 70 + "\n\n")
    
    f.write(f"Total redaction boxes detected: {len(boxes)}\n\n")
    
    f.write("ALL BOXES WITH POSITIONS:\n")
    f.write("-" * 70 + "\n")
    for i, box in enumerate(boxes):
        f.write(f"{i+1}. {box['uuid']}\n")
        f.write(f"   Page: {box['page_index'] + 1}, Line: {box['line_number']}\n")
        f.write(f"   Position: ({box['x_pt']:.1f}, {box['y_pt']:.1f}) points\n")
        f.write(f"   Dimensions: {box['width_pix']}×{box['height_pix']} pixels\n")
        f.write(f"   Width range (pts): {box['x_pt']:.1f} to {box['x_pt'] + box['width_pt']:.1f}\n\n")

print(f"✓ Full analysis written to: {output_path}")

# Print to console also for quick reference
print("\nALL DETECTED BOXES:")
print("-" * 70)
for i, box in enumerate(boxes):
    print(f"{i+1}. UUID: {box['uuid']} | Pos: ({box['x_pt']:.0f},{box['y_pt']:.0f})pts | Size: {box['width_pix']}×{box['height_pix']}px")

In [ ]:
# Re-extract boxes with improved filtering to exclude punctuation and catch all real boxes
# New parameters:
# - black_threshold=235: Balance between catching grey borders (200) vs filtering text (240)
# - min_width=30: Allows smaller boxes like the final redaction
# - min_height=30: Filters out punctuation marks (typically <20px tall)
# - min_aspect_ratio=0.80: Allows more square-ish boxes
boxes = extract_redaction_boxes(DOC, black_threshold=230, min_width=30, min_height=30, min_aspect_ratio=0.80, visualize=True, output_dir="overlays")

print(f"\n✓ Re-extracted {len(boxes)} redaction boxes (punctuation filtered out)")
print(f"✓ Check visualization: overlays/redaction_boxes_page_1.png")

# Save boxes to JSON for use by other scripts (e.g., kellen_render.py)
import json
boxes_save_path = 'redaction_boxes.json'

# Convert numpy types to native Python types for JSON serialization
boxes_serializable = []
for box in boxes:
    box_copy = {}
    for key, value in box.items():
        if isinstance(value, (np.integer, np.floating)):
            box_copy[key] = value.item()
        else:
            box_copy[key] = value
    boxes_serializable.append(box_copy)

with open(boxes_save_path, 'w') as f:
    json.dump(boxes_serializable, f, indent=2)
print(f"✓ Saved {len(boxes)} boxes to: {boxes_save_path}")